In [34]:
import math, typing as t
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)

# ---------- Config ----------
KEEP_COUNTY = False                      # toggle & justify in write-up
BALANCE_METHOD = 'oversample'            # 'oversample' or 'downsample'
TEST_SIZE = 0.20

# 2-layer defaults
TL_HIDDEN = 64
TL_EPOCHS = 50
TL_LR = 5e-3
BATCH = 128

# deeper nets defaults
DL_EPOCHS = 60
DL_LR_SGD = 3e-3
DL_LR_RMS = 3e-3
RMS_BETA = 0.9
RMS_EPS = 1e-8

# Load, Split, and Balance (1.5 points total)

In [37]:
# --- Load, Clean, Encode, Split, and Balance the Dataset ---

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load the dataset
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

# Load the dataset (update the path below to where your CSV is stored)
# The original path caused a FileNotFoundError, updating to the correct path.
# Assumes the file is directly in your Google Drive's root.
path = '/content/drive/My Drive/acs2017_census_tract_data.csv'
df = pd.read_csv(path)
print("✅ Data loaded successfully from Google Drive!")
print("Shape:", df.shape)
display(df.head())



# ================================================================
# 1) CLEAN, TEMP ENCODE, QUANTIZE TARGET, SPLIT, BALANCE (train only)
#    - dropna
#    - case-insensitive County toggle
#    - robust quartiles with tie guard
#    - stratified split
#    - balance train only (default: oversample to keep information)
# ================================================================

df = df.dropna(axis=0).reset_index(drop=True)
print("After dropna:", df.shape)

target_col = 'ChildPoverty'
assert target_col in df.columns, f"{target_col} not found in columns."

# Drop/keep County (case-insensitive)
maybe_county = [c for c in df.columns if c.lower() == 'county']
X = df.drop(columns=[target_col]).copy()
y_reg = df[target_col].astype(float).values

if not KEEP_COUNTY and len(maybe_county) > 0:
    X = X.drop(columns=maybe_county)
    print("Dropped County:", maybe_county)
else:
    print("Kept County:", maybe_county)

# Identify categorical vs numeric (BEFORE any OHE)
cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"Numeric features: {len(num_cols)} | Categorical features: {len(cat_cols)}")

# Temporary integer encoding for categorical features (Regimes A/B)
label_encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    label_encoders[c] = le

X_np = X.values.astype(float)

# Quartile binning with tie guard -> 4 classes (0..3)
q = np.quantile(y_reg, [0, 0.25, 0.5, 0.75, 1.0])
q = np.unique(q)
while len(q) < 5:  # ensure at least 5 unique edges
    q = np.sort(np.unique(np.append(q, q[-1] + 1e-6)))
y_cls = np.digitize(y_reg, q[1:-1], right=True)  # 0..3
K = int(np.max(y_cls) + 1)
print("Quartile cut points:", q, "| Classes:", K)

# Stratified split 80/20
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X_np, y_cls, test_size=TEST_SIZE, random_state=SEED, stratify=y_cls
)

# One-hot for targets
def one_hot(y, K=None):
    K = K or int(np.max(y)) + 1
    Y = np.zeros((y.size, K), dtype=float)
    Y[np.arange(y.size), y] = 1.0
    return Y

Y_train_oh = one_hot(y_train_cls, K)
Y_test_oh  = one_hot(y_test_cls,  K)

# Balance training only
def balance_train(Xtr, ytr, method='oversample'):
    classes, counts = np.unique(ytr, return_counts=True)
    if method == 'downsample':
        n = counts.min()
        idx_bal = np.concatenate([
            np.random.default_rng(SEED).choice(np.where(ytr == c)[0], n, replace=False)
            for c in classes
        ])
    else:
        n = counts.max()
        idx_bal = np.concatenate([
            np.tile(np.where(ytr == c)[0], int(np.ceil(n/np.sum(ytr==c))))[:n]
            for c in classes
        ])
    rng = np.random.default_rng(SEED)
    rng.shuffle(idx_bal)
    return Xtr[idx_bal], ytr[idx_bal]

X_train_bal, y_train_bal = balance_train(X_train, y_train_cls, method=BALANCE_METHOD)
Y_train_bal_oh = one_hot(y_train_bal, K)

print("Train class counts (balanced):", dict(zip(*np.unique(y_train_bal, return_counts=True))))
print("Test  class counts (natural):", dict(zip(*np.unique(y_test_cls, return_counts=True))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Data loaded successfully from Google Drive!
Shape: (74001, 37)


,TractId,State,County,TotalPop,Men,Women,Hispanic,White,Black,Native,...,Walk,OtherTransp,WorkAtHome,MeanCommute,Employed,PrivateWork,PublicWork,SelfEmployed,FamilyWork,Unemployment
0,1001020100,Alabama,Autauga County,1845,899,946,2.4,86.3,5.2,0.0,...,0.5,0.0,2.1,24.5,881,74.2,21.2,4.5,0.0,4.6
1,1001020200,Alabama,Autauga County,2172,1167,1005,1.1,41.6,54.5,0.0,...,0.0,0.5,0.0,22.2,852,75.9,15.0,9.0,0.0,3.4
2,1001020300,Alabama,Autauga County,3385,1533,1852,8.0,61.4,26.5,0.6,...,1.0,0.8,1.5,23.1,1482,73.3,21.1,4.8,0.7,4.7
3,1001020400,Alabama,Autauga County,4267,2001,2266,9.6,80.3,7.1,0.5,...,1.5,2.9,2.1,25.9,1849,75.8,19.7,4.5,0.0,6.1
4,1001020500,Alabama,Autauga County,9965,5054,4911,0.9,77.5,16.4,0.0,...,0.8,0.3,0.7,21.0,4787,71.4,24.1,4.5,0.0,2.3


After dropna: (72718, 37)
Dropped County: ['County']
Numeric features: 34 | Categorical features: 1
Quartile cut points: [  0.    6.2  16.3  31.6 100. ] | Classes: 4
Train class counts (balanced): {np.int64(0): np.int64(14583), np.int64(1): np.int64(14583), np.int64(2): np.int64(14583), np.int64(3): np.int64(14583)}
Test  class counts (natural): {np.int64(0): np.int64(3646), np.int64(1): np.int64(3634), np.int64(2): np.int64(3630), np.int64(3): np.int64(3634)}


# Pre-processing and Initial Modeling

In [38]:
# ================================================================
# 2) PRE-PROCESSING REGIMES (A/B/C)
#    A: raw (cats=int), no normalization
#    B: normalize numeric only
#    C: normalize numeric + OneHotEncoder(handle_unknown='ignore') for cats
# ================================================================

# helper: column indices
num_idx = [X.columns.get_loc(c) for c in num_cols]
cat_cols_present = [c for c in cat_cols if c in X.columns]
cat_idx = [X.columns.get_loc(c) for c in cat_cols_present]

# Regime A (raw)
X1_train = X_train_bal.copy().astype(float)
X1_test  = X_test.copy().astype(float)

# Regime B (normalize numeric only)
X2_train = X_train_bal.copy().astype(float)
X2_test  = X_test.copy().astype(float)
if len(num_idx) > 0:
    scaler_num = StandardScaler()
    X2_train[:, num_idx] = scaler_num.fit_transform(X2_train[:, num_idx])
    X2_test[:,  num_idx] = scaler_num.transform(X2_test[:,  num_idx])

# Regime C (normalize numeric + OHE categoricals; fit on TRAIN only)
try:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
except TypeError:
    ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')

if len(cat_idx) > 0:
    Xc_train_cats = ohe.fit_transform(X_train_bal[:, cat_idx].astype(int))
    Xc_test_cats  = ohe.transform(X_test[:, cat_idx].astype(int))
else:
    Xc_train_cats = np.empty((X_train_bal.shape[0], 0))
    Xc_test_cats  = np.empty((X_test.shape[0], 0))

Xc_train_nums = X_train_bal[:, num_idx].astype(float) if len(num_idx) > 0 else np.empty((X_train_bal.shape[0], 0))
Xc_test_nums  = X_test[:, num_idx].astype(float)       if len(num_idx) > 0 else np.empty((X_test.shape[0], 0))

scaler_c = StandardScaler()
if Xc_train_nums.shape[1] > 0:
    Xc_train_nums = scaler_c.fit_transform(Xc_train_nums)
    Xc_test_nums  = scaler_c.transform(Xc_test_nums)

X3_train = np.concatenate([Xc_train_nums, Xc_train_cats], axis=1)
X3_test  = np.concatenate([Xc_test_nums,  Xc_test_cats],  axis=1)

print("Shapes:",
      "A", X1_train.shape, X1_test.shape,
      "| B", X2_train.shape, X2_test.shape,
      "| C", X3_train.shape, X3_test.shape)

Shapes: A (58332, 35) (14544, 35) | B (58332, 35) (14544, 35) | C (58332, 86) (14544, 86)


# Modeling

In [32]:
# ================================================================
# 3) CORE MLP (vectorized, Glorot-uniform, sigmoid hidden, softmax+CE)
#    - mini-batches with reproducible shuffles per epoch
#    - logs loss/val_loss, acc/val_acc
#    - tracks avg |grad| for W and b per layer per epoch (averaged across batches)
# ================================================================

def glorot_uniform(fan_in, fan_out, rng=np.random):
    limit = math.sqrt(6.0 / (fan_in + fan_out))
    W = rng.uniform(-limit, limit, size=(fan_in, fan_out)).astype(float)
    b = np.zeros((fan_out,), dtype=float)
    return W, b

def sigmoid(z):  return 1.0/(1.0 + np.exp(-z))
def dsigmoid(a): return a*(1.0 - a)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / (np.sum(e, axis=1, keepdims=True))

def cross_entropy(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

def iterate_minibatches(X, Y, batch_size=128, shuffle=True, seed=SEED):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        bidx = idx[s:e]
        yield X[bidx], Y[bidx]

def acc_from_probs(y_true_cls, y_probs):
    return np.mean(np.argmax(y_probs, axis=1) == y_true_cls)

class MLP:
    def __init__(self, layer_sizes: t.List[int], lr: float=1e-3,
                 optimizer: str='sgd', rms_beta: float=0.9, rms_eps: float=1e-8):
        self.layer_sizes = layer_sizes
        self.L = len(layer_sizes) - 1
        self.lr = lr
        self.opt = optimizer
        self.rms_beta = rms_beta
        self.rms_eps = rms_eps
        self.W, self.b = [], []
        for l in range(self.L):
            Wl, bl = glorot_uniform(layer_sizes[l], layer_sizes[l+1], rng=np.random)
            self.W.append(Wl); self.b.append(bl)
        if self.opt == 'rmsprop':
            self.Eg2_W = [np.zeros_like(Wl) for Wl in self.W]
            self.Eg2_b = [np.zeros_like(bl) for bl in self.b]

    def forward(self, X):
        a = X
        A = [a]; Z = []
        for l in range(self.L):
            z = a @ self.W[l] + self.b[l]
            Z.append(z)
            a = sigmoid(z) if l < self.L - 1 else softmax(z)
            A.append(a)
        return A, Z

    def backward(self, A, Z, Y):
        gW = [None]*self.L; gB = [None]*self.L
        dZ = A[-1] - Y
        gW[-1] = A[-2].T @ dZ / Y.shape[0]
        gB[-1] = np.mean(dZ, axis=0)
        for l in range(self.L-2, -1, -1):
            dA = dZ @ self.W[l+1].T
            dZ = dA * dsigmoid(A[l+1])
            gW[l] = A[l].T @ dZ / Y.shape[0]
            gB[l] = np.mean(dZ, axis=0)
        return gW, gB

    def step(self, gW, gB):
        if self.opt == 'sgd':
            for l in range(self.L):
                self.W[l] -= self.lr * gW[l]
                self.b[l] -= self.lr * gB[l]
        elif self.opt == 'rmsprop':
            for l in range(self.L):
                self.Eg2_W[l] = self.rms_beta*self.Eg2_W[l] + (1-self.rms_beta)*(gW[l]**2)
                self.Eg2_b[l] = self.rms_beta*self.Eg2_b[l] + (1-self.rms_beta)*(gB[l]**2)
                self.W[l] -= self.lr * gW[l] / (np.sqrt(self.Eg2_W[l]) + self.rms_eps)
                self.b[l] -= self.lr * gB[l] / (np.sqrt(self.Eg2_b[l]) + self.rms_eps)

    def train(self, Xtr, Ytr, Xval, Yval, yval_cls,
              epochs=50, batch_size=128, verbose=False, seed=SEED):
        hist = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': [], 'grad_mags_W': [], 'grad_mags_b': []}
        for ep in range(epochs):
            # minibatch loop with reproducible shuffle
            magW_sums = [0.0]*self.L; magB_sums = [0.0]*self.L; mb_count = 0
            for Xb, Yb in iterate_minibatches(Xtr, Ytr, batch_size=batch_size, shuffle=True, seed=seed+ep):
                A, Z = self.forward(Xb)
                gW, gB = self.backward(A, Z, Yb)
                for l in range(self.L):
                    magW_sums[l] += float(np.mean(np.abs(gW[l])))
                    magB_sums[l] += float(np.mean(np.abs(gB[l])))
                mb_count += 1
                self.step(gW, gB)

            # metrics
            A_tr, _ = self.forward(Xtr)
            A_val, _ = self.forward(Xval)
            tr_loss = cross_entropy(Ytr, A_tr[-1])
            vl_loss = cross_entropy(Yval, A_val[-1])
            tr_acc = acc_from_probs(np.argmax(Ytr, axis=1), A_tr[-1])
            vl_acc = acc_from_probs(yval_cls, A_val[-1])

            hist['loss'].append(float(tr_loss))
            hist['val_loss'].append(float(vl_loss))
            hist['acc'].append(float(tr_acc))
            hist['val_acc'].append(float(vl_acc))
            hist['grad_mags_W'].append([magW_sums[l]/max(1, mb_count) for l in range(self.L)])
            hist['grad_mags_b'].append([magB_sums[l]/max(1, mb_count) for l in range(self.L)])

            if verbose and (ep % 10 == 0 or ep == epochs-1):
                print(f"Epoch {ep+1}/{epochs} | loss={tr_loss:.4f} val_loss={vl_loss:.4f} "
                      f"acc={tr_acc:.4f} val_acc={vl_acc:.4f}")
        return hist

    def predict_proba(self, X):
        return self.forward(X)[0][-1]

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# ---------- Training helpers ----------
def run_two_layer(Xtr, Ytr, Xte, yte, hidden=TL_HIDDEN, epochs=TL_EPOCHS, lr=TL_LR, batch=BATCH, title=''):
    net = MLP([Xtr.shape[1], hidden, K], lr=lr, optimizer='sgd')
    hist = net.train(Xtr, Ytr, Xte, Y_test_oh, yte, epochs=epochs, batch_size=batch, verbose=False)

    # Loss plots
    plt.figure()
    plt.plot(hist['loss'], label='train'); plt.plot(hist['val_loss'], label='val')
    plt.xlabel('Epoch'); plt.ylabel('CE Loss'); plt.title(f'2-Layer — {title}'); plt.legend(); plt.show()

    ypred = net.predict(Xte)
    acc = accuracy_score(yte, ypred)
    print(f"{title} — Test Accuracy: {acc:.4f}")
    print(classification_report(yte, ypred, digits=3))
    return acc

def train_with_grad_plots(layer_sizes, Xtr, Ytr, Xte, yte,
                          epochs=DL_EPOCHS, lr=DL_LR_SGD, batch=BATCH, title='', optimizer='sgd'):
    net = MLP(layer_sizes, lr=lr, optimizer=optimizer, rms_beta=RMS_BETA, rms_eps=RMS_EPS)
    hist = net.train(Xtr, Ytr, Xte, Y_test_oh, yte, epochs=epochs, batch_size=batch, verbose=False)

    # Loss plot
    plt.figure()
    plt.plot(hist['loss'], label='train'); plt.plot(hist['val_loss'], label='val')
    plt.xlabel('Epoch'); plt.ylabel('CE Loss'); plt.title(f'Loss — {title}'); plt.legend(); plt.show()

    # Grad magnitude plots (W and b)
    gW = np.array(hist['grad_mags_W'])  # (epochs, L)
    gB = np.array(hist['grad_mags_b'])
    for l in range(gW.shape[1]):
        plt.figure()
        plt.plot(gW[:, l], label='|dW|'); plt.plot(gB[:, l], label='|db|')
        plt.xlabel('Epoch'); plt.ylabel('Avg |grad|'); plt.title(f'Grad Magnitude — Layer {l+1} — {title}')
        plt.legend(); plt.show()

    ypred = net.predict(Xte)
    acc = accuracy_score(yte, ypred)
    print(f"{title} — Test Accuracy: {acc:.4f}")
    print(classification_report(yte, ypred, digits=3))
    return acc

# Extra

In [33]:
# ================================================================
# 4) EXPERIMENTS
#    - 2-layer A/B/C
#    - Use C for deeper models (3/4/5)
#    - RMSProp vs SGD on 5-layer
# ================================================================

# Two-layer A/B/C
acc_A = run_two_layer(X1_train, one_hot(y_train_bal, K), X1_test, y_test_cls,
                      title="A) Raw (cats=int, no norm)")
acc_B = run_two_layer(X2_train, one_hot(y_train_bal, K), X2_test, y_test_cls,
                      title="B) Normalize numeric only")
acc_C = run_two_layer(X3_train, one_hot(y_train_bal, K), X3_test, y_test_cls,
                      title="C) Normalize numeric + OneHot(cats)")

print("=== Two-layer Summary ===")
print({"A_raw": acc_A, "B_norm_numeric": acc_B, "C_norm_plus_onehot": acc_C})

# Use C for the rest
input_dim = X3_train.shape[1]
acc_3L = train_with_grad_plots([input_dim, 128, 64, K],
                               X3_train, one_hot(y_train_bal, K), X3_test, y_test_cls,
                               title="3-Layer (128, 64)", optimizer='sgd')
acc_4L = train_with_grad_plots([input_dim, 256, 128, 64, K],
                               X3_train, one_hot(y_train_bal, K), X3_test, y_test_cls,
                               title="4-Layer (256, 128, 64)", optimizer='sgd')
acc_5L_SGD = train_with_grad_plots([input_dim, 256, 128, 64, 32, K],
                                   X3_train, one_hot(y_train_bal, K), X3_test, y_test_cls,
                                   title="5-Layer (256, 128, 64, 32) — SGD",
                                   optimizer='sgd')

# RMSProp vs SGD on 5-layer
acc_5L_RMS = train_with_grad_plots([input_dim, 256, 128, 64, 32, K],
                                   X3_train, one_hot(y_train_bal, K), X3_test, y_test_cls,
                                   title="5-Layer (256, 128, 64, 32) — RMSProp",
                                   optimizer='rmsprop', lr=DL_LR_RMS)

print("=== 5-Layer: SGD vs RMSProp ===")
print({"5L_SGD": acc_5L_SGD, "5L_RMSProp": acc_5L_RMS})

/tmp/ipython-input-1965608165.py:14: RuntimeWarning: overflow encountered in exp
  def sigmoid(z):  return 1.0/(1.0 + np.exp(-z))


KeyboardInterrupt: 